# 06 · Review of the one-time frozen test evaluation

The first code cell verifies the **committed pretest protocol** and archived final-result hashes. This notebook never opens the original test table or calls prediction. The explicit `make final-eval` command performed the single inference pass after the freeze commit; this repeatable notebook reviews its verified aggregate outputs.

All model/feature/calibration/capacity decisions remain those selected using training/validation. No final-test-driven choice is permitted.

In [ ]:
from pathlib import Path
from readmit_iq.decision_support.freeze import verify_freeze
from readmit_iq.decision_support.final_evaluation import verify_published_results

root = Path.cwd()
lock, spec, freeze_commit = verify_freeze(root, require_models=False)
summary = verify_published_results(root)
print("Committed pretest freeze:", freeze_commit)
print("Original scoring pass:", summary["scoring"]["test_table_loads"], "test-table load")
print("Original prediction calls:", summary["scoring"]["prediction_calls"])
print("This review: zero test-table loads; zero prediction calls.")

In [ ]:
import json
import os
import warnings
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("NUMBA_CACHE_DIR", ".cache/numba")
# SHAP's legacy plotting helpers emit these unrelated import-time notices.
warnings.filterwarnings("ignore", category=PendingDeprecationWarning, module=r"shap\.plots.*")
warnings.filterwarnings("ignore", message="IProgress not found.*", module=r"tqdm.*")

from IPython.display import Image, Markdown, display
from readmit_iq.decision_support.plots import read_table, generate_figures

root = Path.cwd()
assert (root / "configs/phase5.yaml").exists()


## Frozen performance and patient-cluster uncertainty

The primary metric is average precision (AP). Bootstrap intervals resample whole patients, rerank at the fixed 10% capacity, and condition on the fitted models. They exclude model-selection/retraining and temporal/site uncertainty. Challenger results are descriptive; their test ranking cannot change the selected primary.

In [ ]:
for partition in ["validation", "test"]:
    display(Markdown(f"**{partition.title()}**"))
    display(read_table(root, partition, "metrics")[["model", "average_precision", "roc_auc", "brier", "recall", "precision", "specificity", "tp", "fp", "fn", "lift"]])
display(read_table(root, "test", "metric_intervals"))
display(read_table(root, "test", "paired_comparisons"))

## Operational results at the predeclared capacities

Top 10% remains the primary portfolio scenario. The realized risk cutoff is determined from unlabeled test ranks, not optimized against outcomes. The capacity grid and deciles describe the same frozen scores. Unique historical people are deduplicated within each selected set and cannot be added across capacities or scaled into concurrent workload.

In [ ]:
display(read_table(root, "test", "capacity", True)[["capacity", "cutoff", "targeted_encounters", "targeted_patients", "tp", "fn", "recall", "precision", "lift"]])
display(read_table(root, "test", "risk_deciles", True).drop(columns=["model", "partition"]))
display(summary["scenario"])

## Prespecified errors and responsible-ML diagnostics

Apply exactly the validation subgroup definitions, size rules and probability-bias criterion. These diagnostics characterize failures; they do not reopen model or threshold selection. Earlier full-cohort EDA and historical/fragmented capture limit the strength of this held-out evaluation.

In [ ]:
display(read_table(root, "test", "error_profiles", True).drop(columns=["model", "partition"]))
groups = read_table(root, "test", "subgroups", True)
display(groups.loc[groups.group.isin(["prior_inpatient", "gender", "race"]), ["group", "level", "encounters", "positives", "suppressed", "prevalence", "recall", "recall_ci_low", "recall_ci_high", "precision", "probability_bias", "material_probability_bias"]])
display(groups.loc[groups.material_probability_bias.eq(True), ["group", "level", "probability_bias", "probability_bias_ci_low", "probability_bias_ci_high"]])

In [ ]:
# Read only the eight reviewed committed figures. Validation explanations remain validation-only.
figures = sorted((root / "reports/figures/phase5").glob("*.png"))
assert len(figures) == 8
for path in figures:
    display(Image(filename=str(path)))

In [ ]:
metadata = json.loads((root / "reports/modeling/final_model_metadata.json").read_text())
assert metadata["model_refitted_after_test"] is False
assert summary["decision_changes_after_test"] is False
assert metadata["targeting"] == lock["policy"]["targeting"]
display({key: metadata[key] for key in ["version", "primary", "pipeline_sha256", "training_commit", "freeze_commit", "features", "clinical_deployment_validated"]})

## Scope and interpretation

Read the [final result narrative](../reports/modeling/final_test_report.md) and [responsible ML diagnostics](../reports/responsible_ml.md). A surfaced readmission is not a prevented readmission. Contemporary external/temporal and prospective clinical/workflow validation are still needed. No service, dashboard, Docker image or deployment is built in this phase.